In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0
)

print("Model loaded")

Model loaded


In [4]:
import json

with open("properties.json", "r") as file:
    properties = json.load(file)

print("Properties loaded:", len(properties))
print(properties[0])

Properties loaded: 8
{'id': 1, 'location': 'Beirut', 'price': 140000, 'bedrooms': 2, 'bathrooms': 2, 'parking': True, 'size': 120}


In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "properties": {
            "command": "python",
            "args": ["property_server.py"],
            "transport": "stdio",
        }
    }
)

tools = await client.get_tools()

print("MCP tools loaded:")
for tool in tools:
    print(tool.name)

MCP tools loaded:
search_properties
get_property_details


In [8]:
search_tool = next(
    tool for tool in tools
    if tool.name == "search_properties"
)

search_result = await search_tool.ainvoke({
    "location": "Beirut",
    "max_price": 150000,
    "bedrooms": 2
})

print(search_result)

[{'type': 'text', 'text': '{\n  "id": 1,\n  "location": "Beirut",\n  "price": 140000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": true,\n  "size": 120\n}', 'id': 'lc_7cd31f48-5add-41be-a956-e09d9035c954'}, {'type': 'text', 'text': '{\n  "id": 3,\n  "location": "Beirut",\n  "price": 145000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": true,\n  "size": 110\n}', 'id': 'lc_b9b4a820-260b-4318-8ad0-01b9a4bb42a2'}, {'type': 'text', 'text': '{\n  "id": 4,\n  "location": "Beirut",\n  "price": 125000,\n  "bedrooms": 2,\n  "bathrooms": 1,\n  "parking": false,\n  "size": 100\n}', 'id': 'lc_bc0da122-9bc6-4738-af83-a004a86de920'}, {'type': 'text', 'text': '{\n  "id": 8,\n  "location": "Beirut",\n  "price": 150000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": false,\n  "size": 125\n}', 'id': 'lc_06d235c7-3b90-46ee-9b39-2d3697c16f5c'}]


In [10]:
from agents import (
    client_requirements_agent,
    property_search_agent,
    property_analysis_agent,
    recommendation_agent
)

print("Agents loaded")

Agents loaded


In [12]:
client_request = """
I am looking for a 2 bedroom apartment in Beirut.
My maximum budget is $150,000 and I need parking.
"""

requirements = client_requirements_agent(client_request)

print(requirements)

{'location': 'Beirut', 'budget': '$150,000', 'bedrooms': 2, 'parking': True}


In [13]:
properties_found = await property_search_agent(
    tools,
    requirements
)

print(properties_found)

[{'type': 'text', 'text': '{\n  "id": 1,\n  "location": "Beirut",\n  "price": 140000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": true,\n  "size": 120\n}', 'id': 'lc_d2546dc0-b00d-42d3-b92b-a00f485c4653'}, {'type': 'text', 'text': '{\n  "id": 3,\n  "location": "Beirut",\n  "price": 145000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": true,\n  "size": 110\n}', 'id': 'lc_32828873-72d1-4861-93af-6081156df751'}, {'type': 'text', 'text': '{\n  "id": 4,\n  "location": "Beirut",\n  "price": 125000,\n  "bedrooms": 2,\n  "bathrooms": 1,\n  "parking": false,\n  "size": 100\n}', 'id': 'lc_d59da9ed-6c55-4f2a-8b59-d71e6fde1ccb'}, {'type': 'text', 'text': '{\n  "id": 8,\n  "location": "Beirut",\n  "price": 150000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": false,\n  "size": 125\n}', 'id': 'lc_7d15ea0f-3c0b-4a2f-bf4f-1be6952f343f'}]


In [14]:
analysis = property_analysis_agent(
    properties_found,
    requirements
)

print(analysis)

### Property Analysis Report

**Client Requirements:**
*   **Location:** Beirut
*   **Budget:** $\le \$150,000$
*   **Bedrooms:** 2
*   **Parking:** Yes

***

#### Property Comparison

**Property 1 (ID: lc_d2546dc0-b00d-42d3-b92b-a00f485c4653)**
*   **Match Status:** **Excellent Match.**
*   **Analysis:** Meets all requirements. The location is Beirut, the price ($\$140,000$) is within budget, it has 2 bedrooms, and crucially, it includes parking.

**Property 2 (ID: lc_32828873-72d1-4861-93af-6081156df751)**
*   **Match Status:** **Excellent Match.**
*   **Analysis:** Meets all requirements. The location is Beirut, the price ($\$145,000$) is within budget, it has 2 bedrooms, and includes parking.

**Property 3 (ID: lc_d59da9ed-6c55-4f2a-8b59-d71e6fde1ccb)**
*   **Match Status:** **Partial Match (Fails on Parking).**
*   **Analysis:** Meets location, budget, and bedroom count. However, it does **not** include parking, which is a mandatory requirement.

**Property 4 (ID: lc_7d15ea0f-3c0b

In [17]:
recommendation = recommendation_agent(analysis)

print(recommendation)

Based on your requirements, I have narrowed down the options to two excellent choices that perfectly meet all your criteria.

I strongly recommend considering **Property 1** and **Property 2**.

### 🏡 Top Recommendations

**1. Property ID: lc_d2546dc0-b00d-42d3-b92b-a00f485c4653**
*   **Why it's suitable:** This property is an excellent fit. It is located in Beirut, falls comfortably within your budget ($\$140,000$), offers 2 bedrooms, and crucially, includes dedicated parking.

**2. Property ID: lc_32828873-72d1-4861-93af-6081156df751**
*   **Why it's suitable:** This is also an excellent match. It meets all your needs: Beirut location, 2 bedrooms, included parking, and a price ($\$145,000$) that is well within your maximum budget.

***

**Summary:** Both properties are ideal because they satisfy all four of your mandatory requirements: Location (Beirut), Budget ($\le \$150,000$), Bedrooms (2), and Parking (Yes). I recommend reviewing both to see which one best fits your lifestyle pre

In [18]:
from graph import graph_builder

graph = graph_builder.compile()

result = await graph.ainvoke({
    "client_request": client_request,
    "tools": tools
})

print(result["recommendation"])

Based on the detailed analysis, I have two excellent recommendations for you. Both properties perfectly meet all of your mandatory requirements.

### 🏡 Final Recommendation

I strongly recommend considering **Property 1** and **Property 2**.

**Why these properties are suitable:**

*   **Perfect Match:** Both properties are located in Beirut, fit within your budget, and offer the required 2 bedrooms.
*   **Crucial Feature:** Most importantly, both listings include **parking**, which was a mandatory requirement for your search.

**Comparison:**

*   **Property 1:** This is the most budget-friendly option at **$140,000**, while still providing all the necessary amenities.
*   **Property 2:** This option is slightly higher at **$145,000**, but also provides the full set of features and convenience you are looking for.

***

*We have successfully filtered out the other properties because they failed to include the mandatory parking requirement.*


In [19]:
def safety_check(properties, requirements):
    budget = int(str(requirements["budget"]).replace("$", "").replace(",", ""))
    bedrooms = int(requirements["bedrooms"])

    safe_properties = []

    for property in properties:
        if (
            property["location"].lower() == requirements["location"].lower()
            and property["price"] <= budget
            and property["bedrooms"] == bedrooms
            and (not requirements["parking"] or property["parking"])
        ):
            safe_properties.append(property)

    return safe_properties


safe_properties = safety_check(
    properties,
    requirements
)

print("Properties passing safety check:")
print(safe_properties)

Properties passing safety check:
[{'id': 1, 'location': 'Beirut', 'price': 140000, 'bedrooms': 2, 'bathrooms': 2, 'parking': True, 'size': 120}, {'id': 3, 'location': 'Beirut', 'price': 145000, 'bedrooms': 2, 'bathrooms': 2, 'parking': True, 'size': 110}]


In [20]:
test_properties = [
    {
        "id": 99,
        "location": "Beirut",
        "price": 180000,
        "bedrooms": 2,
        "bathrooms": 2,
        "parking": True,
        "size": 150
    },
    {
        "id": 100,
        "location": "Beirut",
        "price": 140000,
        "bedrooms": 3,
        "bathrooms": 2,
        "parking": True,
        "size": 150
    },
    {
        "id": 101,
        "location": "Beirut",
        "price": 140000,
        "bedrooms": 2,
        "bathrooms": 2,
        "parking": False,
        "size": 120
    }
]

safety_test = safety_check(
    test_properties,
    requirements
)

print("Properties passing safety check:")
print(safety_test)

Properties passing safety check:
[]


In [21]:
no_match_request = """
I am looking for a 4 bedroom apartment in Beirut.
My maximum budget is $100,000 and I need parking.
"""

no_match_requirements = client_requirements_agent(no_match_request)

no_match_properties = await property_search_agent(
    tools,
    no_match_requirements
)

no_match_safe = safety_check(
    no_match_properties,
    no_match_requirements
)

print("Requirements:")
print(no_match_requirements)

print("\nSafe properties:")
print(no_match_safe)

Requirements:
{'location': 'Beirut', 'budget': '$100,000', 'bedrooms': 4, 'parking': True}

Safe properties:
[]


In [22]:
test_cases = [
    """
    I want a 2 bedroom apartment in Beirut.
    My maximum budget is $150,000 and I need parking.
    """,
    
    """
    I want a 3 bedroom apartment in Jounieh.
    My maximum budget is $170,000 and I need parking.
    """,
    
    """
    I want a 2 bedroom apartment in Beirut.
    My maximum budget is $130,000 and I do not need parking.
    """
]

for i, request in enumerate(test_cases, 1):
    print(f"\nTest case {i}")
    print("Request:", request.strip())

    test_requirements = client_requirements_agent(request)

    test_properties = await property_search_agent(
        tools,
        test_requirements
    )

    test_safe = safety_check(
        test_properties,
        test_requirements
    )

    print("Requirements:", test_requirements)
    print("Safe properties:", test_safe)


Test case 1
Request: I want a 2 bedroom apartment in Beirut.
    My maximum budget is $150,000 and I need parking.


KeyError: 'location'

In [30]:
print("Last requirements:")
print(test_requirements)

print("\nLast properties:")
print(test_properties)

Last requirements:
{'location': 'Beirut', 'budget': '$150,000', 'bedrooms': 2, 'parking': True}

Last properties:
[{'type': 'text', 'text': '{\n  "id": 1,\n  "location": "Beirut",\n  "price": 140000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": true,\n  "size": 120\n}', 'id': 'lc_138a88b3-77c8-44c8-a267-88d0afc29659'}, {'type': 'text', 'text': '{\n  "id": 3,\n  "location": "Beirut",\n  "price": 145000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": true,\n  "size": 110\n}', 'id': 'lc_6ded774a-c082-4d17-9477-cb45c65b1f0b'}, {'type': 'text', 'text': '{\n  "id": 4,\n  "location": "Beirut",\n  "price": 125000,\n  "bedrooms": 2,\n  "bathrooms": 1,\n  "parking": false,\n  "size": 100\n}', 'id': 'lc_e7bb3d31-037a-4f8f-9654-6796ec93a1a8'}, {'type': 'text', 'text': '{\n  "id": 8,\n  "location": "Beirut",\n  "price": 150000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": false,\n  "size": 125\n}', 'id': 'lc_bc1d6d25-577d-479b-b903-7fa4066b583b'}]


In [49]:
import importlib
import agents
import inspect

importlib.reload(agents)

property_search_agent = agents.property_search_agent

print(inspect.getsource(property_search_agent))

async def property_search_agent(tools, requirements):
    budget = requirements["budget"]
    budget = int(str(budget).replace("$", "").replace(",", ""))

    search_tool = next(
        tool for tool in tools
        if tool.name == "search_properties"
    )

    result = await search_tool.ainvoke({
        "location": requirements["location"],
        "max_price": budget,
        "bedrooms": int(requirements["bedrooms"])
    })

    properties = []

    for item in result:
        if item.get("type") == "text":
            properties.append(json.loads(item["text"]))

    return properties



In [51]:
properties_found = await property_search_agent(
    tools,
    requirements
)

print(properties_found)

[{'id': 1, 'location': 'Beirut', 'price': 140000, 'bedrooms': 2, 'bathrooms': 2, 'parking': True, 'size': 120}, {'id': 3, 'location': 'Beirut', 'price': 145000, 'bedrooms': 2, 'bathrooms': 2, 'parking': True, 'size': 110}, {'id': 4, 'location': 'Beirut', 'price': 125000, 'bedrooms': 2, 'bathrooms': 1, 'parking': False, 'size': 100}, {'id': 8, 'location': 'Beirut', 'price': 150000, 'bedrooms': 2, 'bathrooms': 2, 'parking': False, 'size': 125}]


In [53]:
safe_properties = safety_check(
    properties_found,
    requirements
)

print("Properties passing safety check:")
print(safe_properties)

Properties passing safety check:
[{'id': 1, 'location': 'Beirut', 'price': 140000, 'bedrooms': 2, 'bathrooms': 2, 'parking': True, 'size': 120}, {'id': 3, 'location': 'Beirut', 'price': 145000, 'bedrooms': 2, 'bathrooms': 2, 'parking': True, 'size': 110}]


In [32]:
async def property_search_agent(tools, requirements):
    budget = requirements["budget"]
    budget = int(str(budget).replace("$", "").replace(",", ""))

    search_tool = next(
        tool for tool in tools
        if tool.name == "search_properties"
    )

    result = await search_tool.ainvoke({
        "location": requirements["location"],
        "max_price": budget,
        "bedrooms": int(requirements["bedrooms"])
    })

    properties = []

    for item in result:
        if item.get("type") == "text":
            properties.append(json.loads(item["text"]))

    return properties

In [34]:
import importlib
import agents

importlib.reload(agents)

from agents import (
    client_requirements_agent,
    property_search_agent,
    property_analysis_agent,
    recommendation_agent
)

print("Agents reloaded")

Agents reloaded


In [36]:
properties_found = await property_search_agent(
    tools,
    requirements
)

print(properties_found)

[{'type': 'text', 'text': '{\n  "id": 1,\n  "location": "Beirut",\n  "price": 140000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": true,\n  "size": 120\n}', 'id': 'lc_b669e7e2-ca25-4eeb-b8f5-a0fab7758604'}, {'type': 'text', 'text': '{\n  "id": 3,\n  "location": "Beirut",\n  "price": 145000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": true,\n  "size": 110\n}', 'id': 'lc_788ba4ee-0688-4063-80fd-b41609399726'}, {'type': 'text', 'text': '{\n  "id": 4,\n  "location": "Beirut",\n  "price": 125000,\n  "bedrooms": 2,\n  "bathrooms": 1,\n  "parking": false,\n  "size": 100\n}', 'id': 'lc_7e4653d0-8ce0-4716-86dd-5352392902ab'}, {'type': 'text', 'text': '{\n  "id": 8,\n  "location": "Beirut",\n  "price": 150000,\n  "bedrooms": 2,\n  "bathrooms": 2,\n  "parking": false,\n  "size": 125\n}', 'id': 'lc_f7cdfd52-e331-43b8-be21-211d1b4af134'}]


In [43]:
import inspect

print(inspect.getsource(property_search_agent))

async def property_search_agent(tools, requirements):
    budget = requirements["budget"]
    budget = int(str(budget).replace("$", "").replace(",", ""))

    result = await tools[0].ainvoke({
        "location": requirements["location"],
        "max_price": budget,
        "bedrooms": int(requirements["bedrooms"])
    })

    return result



In [45]:
analysis = property_analysis_agent(
    properties_found,
    requirements
)

print(analysis)

KeyboardInterrupt: 

In [ ]:
recommendation = recommendation_agent(analysis)

print(recommendation)

In [ ]:
from graph import graph_builder

graph = graph_builder.compile()

result = await graph.ainvoke({
    "client_request": client_request,
    "tools": tools
})

print(result["recommendation"])

In [56]:
import importlib
import graph

importlib.reload(graph)

graph_builder = graph.graph_builder

print("Graph reloaded")

Graph reloaded


In [58]:
graph = graph_builder.compile()

result = await graph.ainvoke({
    "client_request": client_request,
    "tools": tools
})

print(result["recommendation"])

Based on the analysis, I highly recommend both **Property ID 1** and **Property ID 3**.

Both properties are excellent matches and meet all of your stated requirements.

**Why they are suitable:**

*   **Location:** Both are located in Beirut.
*   **Features:** Both offer 2 bedrooms and include parking.
*   **Budget:** Both prices are comfortably within your specified budget.

**Recommendation Summary:**

*   **Property ID 1:** This is the most budget-friendly option at **$140,000**, while still providing all the necessary features.
*   **Property ID 3:** This is also an excellent option at **$145,000**.

Since both properties are nearly identical in terms of suitability, I recommend reviewing both to see which specific unit or price point best fits your immediate needs.


In [60]:
no_match_request = """
I am looking for a 4 bedroom apartment in Beirut.
My maximum budget is $100,000 and I need parking.
"""

no_match_result = await graph.ainvoke({
    "client_request": no_match_request,
    "tools": tools
})

print(no_match_result["recommendation"])

**To provide you with the best possible recommendation, I need the property data.**

The analysis provided indicates that no properties were included for comparison. Once you provide the list of properties, I will immediately review them against your requirements and give you a clear, final recommendation.

***

**Please provide the properties you would like me to analyze, and I will complete the recommendation for you!**


In [63]:
import importlib
import agents

importlib.reload(agents)

property_analysis_agent = agents.property_analysis_agent

print("Agents reloaded")

Agents reloaded


In [65]:
no_match_result = await graph.ainvoke({
    "client_request": no_match_request,
    "tools": tools
})

print(no_match_result["recommendation"])

**To provide you with the best possible recommendation, I need the property data.**

The analysis provided indicates that no properties were included for comparison. Once you provide the list of properties, I will immediately review them against your requirements and give you a clear, final recommendation.

***

**Please provide the properties you would like me to analyze, and I will complete the recommendation for you!**


In [67]:
import importlib
import graph

importlib.reload(graph)

graph_builder = graph.graph_builder

print("Graph reloaded")

Graph reloaded


In [72]:
no_match_result = await graph.ainvoke({
    "client_request": no_match_request,
    "tools": tools
})

print(no_match_result["recommendation"])

AttributeError: module 'graph' has no attribute 'ainvoke'

In [74]:
graph_builder = graph.graph_builder
graph = graph_builder.compile()

print("Graph compiled")

Graph compiled


In [76]:
no_match_result = await graph.ainvoke({
    "client_request": no_match_request,
    "tools": tools
})

print(no_match_result["recommendation"])

Based on the property analysis, there are currently no properties that meet all of your specified requirements.

To help you find the perfect home, I recommend we adjust the search criteria. Please let me know if you are flexible on any of the following:

*   **Location:** Are there nearby neighborhoods you would consider?
*   **Budget:** Is there a slight range of price you could accommodate?
*   **Features:** Are there any non-essential features (e.g., a specific type of yard, number of bathrooms) that could be removed from the criteria?

Once we adjust the parameters, I can immediately search for the best available options.


In [79]:
import importlib
import agents
import graph

importlib.reload(agents)
importlib.reload(graph)

graph_builder = graph.graph_builder
graph = graph_builder.compile()

print("Agents and graph reloaded")

Agents and graph reloaded


In [81]:
no_match_result = await graph.ainvoke({
    "client_request": no_match_request,
    "tools": tools
})

print(no_match_result["recommendation"])

No properties match the client's requirements.


In [83]:
for i, request in enumerate(test_cases, 1):
    result = await graph.ainvoke({
        "client_request": request,
        "tools": tools
    })

    print("\nTest case", i)
    print(result["recommendation"])


Test case 1
Based on the analysis, I highly recommend both **Property ID 1** and **Property ID 3**.

Both properties are excellent matches and meet all of your stated requirements.

**Why they are suitable:**

*   **Location:** Both are located in Beirut.
*   **Features:** Both offer 2 bedrooms and include parking.
*   **Budget:** Both prices are comfortably within your specified budget.

**Recommendation Summary:**

*   **Property ID 1:** This is the most budget-friendly option at **$140,000**, while still providing all the necessary features.
*   **Property ID 3:** This is also an excellent option at **$145,000**.

Since both properties are nearly identical in terms of suitability, I recommend reviewing both to see which specific unit or price point best fits your immediate needs.

Test case 2
Based on the detailed analysis, I have an excellent recommendation for you.

**Final Recommendation:**

We strongly recommend **Property ID 6**.

**Why it is suitable:**

This property is an *